# face_qef GPU Compare

JIT compile current `intersect_qef`, `face_qef_old`, `face_qef_ref`, current `face_qef`, and a CPU `face_qef` wrapper. `intersect_qef` only prepares the active voxel and brick inputs used by the face QEF variants.

In [1]:
import os
import sys
import time
import tempfile
import statistics
from pathlib import Path

CONDA_PREFIX = Path(os.environ.get('CONDA_PREFIX', '/home/quanta/.conda/envs/symm-enforce'))
os.environ['PATH'] = f"{CONDA_PREFIX / 'bin'}:{os.environ.get('PATH', '')}"
os.environ['CUDA_HOME'] = str(CONDA_PREFIX)
os.environ['CC'] = str(CONDA_PREFIX / 'bin/gcc')
os.environ['CXX'] = str(CONDA_PREFIX / 'bin/g++')
os.environ['MAX_JOBS'] = str(os.cpu_count() or 1)

import torch
import torch.utils.cpp_extension as cpp_extension
from torch.utils.cpp_extension import load

cpp_extension.CUDA_HOME = str(CONDA_PREFIX)
torch.set_grad_enabled(False)

ROOT = Path.cwd()
if not (ROOT / 'setup.py').exists():
    ROOT = ROOT.parent
assert (ROOT / 'setup.py').exists(), f'Cannot find repo root from {Path.cwd()}'

DEVICE = torch.device('cuda')
assert torch.cuda.is_available(), 'CUDA is required for this notebook'

GRID_SIZE = int(os.environ.get('FDG_GRID_SIZE', '512'))
CHUNK_TRIANGLES = int(os.environ.get('FDG_CHUNK_TRIANGLES', '20000'))
SUBDIVIDE_AREA_THRESHOLD = float(os.environ.get('FDG_SUBDIVIDE_AREA_THRESHOLD', str(1.0 / (GRID_SIZE * GRID_SIZE))))
SUBDIVIDE_MAX_ITERS = int(os.environ.get('FDG_SUBDIVIDE_MAX_ITERS', '12'))
WARM_REPEATS = int(os.environ.get('FDG_WARM_REPEATS', '10'))
RUN_CPU_SUBDIVIDED = os.environ.get('FDG_RUN_CPU_SUBDIVIDED', '1') != '0'

print('repo:', ROOT)
print('python:', sys.executable)
print('torch:', torch.__version__, 'cuda:', torch.version.cuda)
print('device:', torch.cuda.get_device_name(DEVICE))
print('CONDA_PREFIX:', CONDA_PREFIX)
print('nvcc:', CONDA_PREFIX / 'bin/nvcc')
print('gcc:', os.environ['CC'])
print('g++:', os.environ['CXX'])
print('MAX_JOBS:', os.environ['MAX_JOBS'])
print('GRID_SIZE:', GRID_SIZE, 'CHUNK_TRIANGLES:', CHUNK_TRIANGLES)
print('SUBDIVIDE_AREA_THRESHOLD:', SUBDIVIDE_AREA_THRESHOLD, 'SUBDIVIDE_MAX_ITERS:', SUBDIVIDE_MAX_ITERS)
print('WARM_REPEATS:', WARM_REPEATS, 'RUN_CPU_SUBDIVIDED:', RUN_CPU_SUBDIVIDED)


repo: /mnt/nvmefs/Projects/o-voxel-gpu
python: /home/quanta/.conda/envs/symm-enforce/bin/python
torch: 2.6.0+cu124 cuda: 12.4
device: NVIDIA GeForce RTX 4090
CONDA_PREFIX: /home/quanta/.conda/envs/symm-enforce
nvcc: /home/quanta/.conda/envs/symm-enforce/bin/nvcc
gcc: /home/quanta/.conda/envs/symm-enforce/bin/gcc
g++: /home/quanta/.conda/envs/symm-enforce/bin/g++
MAX_JOBS: 32
GRID_SIZE: 512 CHUNK_TRIANGLES: 20000
SUBDIVIDE_AREA_THRESHOLD: 3.814697265625e-06 SUBDIVIDE_MAX_ITERS: 12
WARM_REPEATS: 10 RUN_CPU_SUBDIVIDED: True


## JIT Extension

The temporary C++ binding includes the CPU `face_qef` implementation from the current CPU source and binds the three GPU face QEF variants from the CUDA module.

In [2]:
build_dir = Path(tempfile.mkdtemp(prefix='fdg_face_qef_jit_'))
binding_path = build_dir / 'binding.cpp'
probe_path = build_dir / 'probe.cu'
torch_build_dir = build_dir / 'torch_ext'
torch_build_dir.mkdir(parents=True, exist_ok=True)

binding_cpp = r"""
#include <torch/extension.h>
#include <Eigen/Dense>
#include <cmath>
#include <unordered_map>
#include <vector>
#include <tuple>
#include "__ROOT__/src/convert/flexible_dual_grid.cpp"
#include "__ROOT__/src/convert/mesh_to_flexible_dual_grid_gpu/api.h"

namespace py = pybind11;

std::tuple<torch::Tensor, torch::Tensor, torch::Tensor, torch::Tensor, torch::Tensor> face_qef_gpu_probe(
    const torch::Tensor& triangles,
    const torch::Tensor& voxel_size,
    const torch::Tensor& grid_range,
    const torch::Tensor& voxel);
torch::Tensor face_qef_gpu_normal_probe(
    const torch::Tensor& triangles,
    int64_t tri_id);

namespace {

torch::Tensor face_qef_cpu_only(
    const torch::Tensor& triangles,
    const torch::Tensor& voxel_size,
    const torch::Tensor& grid_range,
    const torch::Tensor& voxels) {
    const int64_t T = triangles.size(0);
    const int64_t N = voxels.size(0);
    const float* tri_ptr = triangles.data_ptr<float>();
    const float* vs = voxel_size.data_ptr<float>();
    const int32_t* gr = grid_range.data_ptr<int32_t>();
    const int32_t* vox_ptr = voxels.data_ptr<int32_t>();

    std::vector<Eigen::Vector3f> tri_vec;
    tri_vec.reserve(static_cast<size_t>(T) * 3);
    for (int64_t t = 0; t < T; ++t) {
        tri_vec.emplace_back(tri_ptr[9 * t + 0], tri_ptr[9 * t + 1], tri_ptr[9 * t + 2]);
        tri_vec.emplace_back(tri_ptr[9 * t + 3], tri_ptr[9 * t + 4], tri_ptr[9 * t + 5]);
        tri_vec.emplace_back(tri_ptr[9 * t + 6], tri_ptr[9 * t + 7], tri_ptr[9 * t + 8]);
    }

    std::unordered_map<VoxelCoord, size_t> hash_table;
    hash_table.reserve(static_cast<size_t>(N) * 2);
    for (int64_t i = 0; i < N; ++i) {
        hash_table[VoxelCoord{vox_ptr[3 * i + 0], vox_ptr[3 * i + 1], vox_ptr[3 * i + 2]}] = static_cast<size_t>(i);
    }

    std::vector<Eigen::Matrix4f> qefs(static_cast<size_t>(N), Eigen::Matrix4f::Zero());
    ::face_qef(
        Eigen::Vector3f(vs[0], vs[1], vs[2]),
        Eigen::Vector3i(gr[0], gr[1], gr[2]),
        Eigen::Vector3i(gr[3], gr[4], gr[5]),
        tri_vec,
        hash_table,
        qefs);

    auto out = torch::empty({N, 10}, torch::TensorOptions().dtype(torch::kFloat32).device(torch::kCPU));
    float* out_ptr = out.data_ptr<float>();
    for (int64_t i = 0; i < N; ++i) {
        const Eigen::Matrix4f& Q = qefs[static_cast<size_t>(i)];
        out_ptr[10 * i + 0] = Q(0, 0);
        out_ptr[10 * i + 1] = Q(0, 1);
        out_ptr[10 * i + 2] = Q(0, 2);
        out_ptr[10 * i + 3] = Q(0, 3);
        out_ptr[10 * i + 4] = Q(1, 1);
        out_ptr[10 * i + 5] = Q(1, 2);
        out_ptr[10 * i + 6] = Q(1, 3);
        out_ptr[10 * i + 7] = Q(2, 2);
        out_ptr[10 * i + 8] = Q(2, 3);
        out_ptr[10 * i + 9] = Q(3, 3);
    }
    return out;
}

std::tuple<torch::Tensor, torch::Tensor, torch::Tensor, torch::Tensor> face_qef_cpu_probe(
    const torch::Tensor& triangles,
    const torch::Tensor& voxel_size,
    const torch::Tensor& grid_range,
    const torch::Tensor& voxel) {
    const int64_t T = triangles.size(0);
    const float* tri_ptr = triangles.data_ptr<float>();
    const float* vs = voxel_size.data_ptr<float>();
    const int32_t* gr = grid_range.data_ptr<int32_t>();
    const int32_t* vp = voxel.data_ptr<int32_t>();
    const int x = vp[0];
    const int y = vp[1];
    const int z = vp[2];
    const Eigen::Vector3f e_voxel_size(vs[0], vs[1], vs[2]);
    const Eigen::Vector3i grid_min(gr[0], gr[1], gr[2]);
    const Eigen::Vector3i grid_max(gr[3], gr[4], gr[5]);
    const Eigen::Vector3f p = e_voxel_size.cwiseProduct(Eigen::Vector3f(x, y, z));

    auto hit = torch::empty({T}, torch::TensorOptions().dtype(torch::kUInt8).device(torch::kCPU));
    auto fail_code = torch::empty({T}, torch::TensorOptions().dtype(torch::kInt32).device(torch::kCPU));
    auto fail_value = torch::empty({T}, torch::TensorOptions().dtype(torch::kFloat32).device(torch::kCPU));
    auto normal_trace = torch::empty({T}, torch::TensorOptions().dtype(torch::kFloat32).device(torch::kCPU));
    uint8_t* hit_ptr = hit.data_ptr<uint8_t>();
    int32_t* code_ptr = fail_code.data_ptr<int32_t>();
    float* val_ptr = fail_value.data_ptr<float>();
    float* trace_ptr = normal_trace.data_ptr<float>();

    for (int64_t t = 0; t < T; ++t) {
        const Eigen::Vector3f v0(tri_ptr[9 * t + 0], tri_ptr[9 * t + 1], tri_ptr[9 * t + 2]);
        const Eigen::Vector3f v1(tri_ptr[9 * t + 3], tri_ptr[9 * t + 4], tri_ptr[9 * t + 5]);
        const Eigen::Vector3f v2(tri_ptr[9 * t + 6], tri_ptr[9 * t + 7], tri_ptr[9 * t + 8]);
        const Eigen::Vector3f e0 = v1 - v0;
        const Eigen::Vector3f e1 = v2 - v1;
        const Eigen::Vector3f e2 = v0 - v2;
        const Eigen::Vector3f n = e0.cross(e1).normalized();
        trace_ptr[t] = n.squaredNorm();

        const Eigen::Vector3f bb_min_f = v0.cwiseMin(v1).cwiseMin(v2).cwiseQuotient(e_voxel_size);
        const Eigen::Vector3f bb_max_f = v0.cwiseMax(v1).cwiseMax(v2).cwiseQuotient(e_voxel_size);
        const Eigen::Vector3i bb_min(std::max(static_cast<int>(bb_min_f.x()), grid_min.x()),
                                     std::max(static_cast<int>(bb_min_f.y()), grid_min.y()),
                                     std::max(static_cast<int>(bb_min_f.z()), grid_min.z()));
        const Eigen::Vector3i bb_max(std::min(static_cast<int>(bb_max_f.x() + 1), grid_max.x()),
                                     std::min(static_cast<int>(bb_max_f.y() + 1), grid_max.y()),
                                     std::min(static_cast<int>(bb_max_f.z() + 1), grid_max.z()));
        if (x < bb_min.x() || x >= bb_max.x() || y < bb_min.y() || y >= bb_max.y() || z < bb_min.z() || z >= bb_max.z()) {
            hit_ptr[t] = 0;
            code_ptr[t] = 1;
            val_ptr[t] = 0.0f;
            continue;
        }

        const Eigen::Vector3f c(n.x() > 0.0f ? e_voxel_size.x() : 0.0f,
                                n.y() > 0.0f ? e_voxel_size.y() : 0.0f,
                                n.z() > 0.0f ? e_voxel_size.z() : 0.0f);
        const float d1 = n.dot(c - v0);
        const float d2 = n.dot(e_voxel_size - c - v0);
        const float n_dot_p = n.dot(p);
        float value = (n_dot_p + d1) * (n_dot_p + d2);
        if (value > 0.0f) { hit_ptr[t] = 0; code_ptr[t] = 2; val_ptr[t] = value; continue; }

        const int mul_xy = n.z() < 0.0f ? -1 : 1;
        const Eigen::Vector2f n_xy_e0(-mul_xy * e0.y(), mul_xy * e0.x());
        const Eigen::Vector2f n_xy_e1(-mul_xy * e1.y(), mul_xy * e1.x());
        const Eigen::Vector2f n_xy_e2(-mul_xy * e2.y(), mul_xy * e2.x());
        const float d_xy_e0 = -n_xy_e0.dot(v0.head<2>()) + n_xy_e0.cwiseMax(0.0f).dot(e_voxel_size.head<2>());
        const float d_xy_e1 = -n_xy_e1.dot(v1.head<2>()) + n_xy_e1.cwiseMax(0.0f).dot(e_voxel_size.head<2>());
        const float d_xy_e2 = -n_xy_e2.dot(v2.head<2>()) + n_xy_e2.cwiseMax(0.0f).dot(e_voxel_size.head<2>());
        const Eigen::Vector2f p_xy(p.x(), p.y());
        value = n_xy_e0.dot(p_xy) + d_xy_e0;
        if (value < 0.0f) { hit_ptr[t] = 0; code_ptr[t] = 3; val_ptr[t] = value; continue; }
        value = n_xy_e1.dot(p_xy) + d_xy_e1;
        if (value < 0.0f) { hit_ptr[t] = 0; code_ptr[t] = 4; val_ptr[t] = value; continue; }
        value = n_xy_e2.dot(p_xy) + d_xy_e2;
        if (value < 0.0f) { hit_ptr[t] = 0; code_ptr[t] = 5; val_ptr[t] = value; continue; }

        const int mul_yz = n.x() < 0.0f ? -1 : 1;
        const Eigen::Vector2f n_yz_e0(-mul_yz * e0.z(), mul_yz * e0.y());
        const Eigen::Vector2f n_yz_e1(-mul_yz * e1.z(), mul_yz * e1.y());
        const Eigen::Vector2f n_yz_e2(-mul_yz * e2.z(), mul_yz * e2.y());
        const float d_yz_e0 = -n_yz_e0.dot(Eigen::Vector2f(v0.y(), v0.z())) + n_yz_e0.cwiseMax(0.0f).dot(Eigen::Vector2f(e_voxel_size.y(), e_voxel_size.z()));
        const float d_yz_e1 = -n_yz_e1.dot(Eigen::Vector2f(v1.y(), v1.z())) + n_yz_e1.cwiseMax(0.0f).dot(Eigen::Vector2f(e_voxel_size.y(), e_voxel_size.z()));
        const float d_yz_e2 = -n_yz_e2.dot(Eigen::Vector2f(v2.y(), v2.z())) + n_yz_e2.cwiseMax(0.0f).dot(Eigen::Vector2f(e_voxel_size.y(), e_voxel_size.z()));
        const Eigen::Vector2f p_yz(p.y(), p.z());
        value = n_yz_e0.dot(p_yz) + d_yz_e0;
        if (value < 0.0f) { hit_ptr[t] = 0; code_ptr[t] = 6; val_ptr[t] = value; continue; }
        value = n_yz_e1.dot(p_yz) + d_yz_e1;
        if (value < 0.0f) { hit_ptr[t] = 0; code_ptr[t] = 7; val_ptr[t] = value; continue; }
        value = n_yz_e2.dot(p_yz) + d_yz_e2;
        if (value < 0.0f) { hit_ptr[t] = 0; code_ptr[t] = 8; val_ptr[t] = value; continue; }

        const int mul_zx = n.y() < 0.0f ? -1 : 1;
        const Eigen::Vector2f n_zx_e0(-mul_zx * e0.x(), mul_zx * e0.z());
        const Eigen::Vector2f n_zx_e1(-mul_zx * e1.x(), mul_zx * e1.z());
        const Eigen::Vector2f n_zx_e2(-mul_zx * e2.x(), mul_zx * e2.z());
        const float d_zx_e0 = -n_zx_e0.dot(Eigen::Vector2f(v0.z(), v0.x())) + n_zx_e0.cwiseMax(0.0f).dot(Eigen::Vector2f(e_voxel_size.z(), e_voxel_size.x()));
        const float d_zx_e1 = -n_zx_e1.dot(Eigen::Vector2f(v1.z(), v1.x())) + n_zx_e1.cwiseMax(0.0f).dot(Eigen::Vector2f(e_voxel_size.z(), e_voxel_size.x()));
        const float d_zx_e2 = -n_zx_e2.dot(Eigen::Vector2f(v2.z(), v2.x())) + n_zx_e2.cwiseMax(0.0f).dot(Eigen::Vector2f(e_voxel_size.z(), e_voxel_size.x()));
        const Eigen::Vector2f p_zx(p.z(), p.x());
        value = n_zx_e0.dot(p_zx) + d_zx_e0;
        if (value < 0.0f) { hit_ptr[t] = 0; code_ptr[t] = 9; val_ptr[t] = value; continue; }
        value = n_zx_e1.dot(p_zx) + d_zx_e1;
        if (value < 0.0f) { hit_ptr[t] = 0; code_ptr[t] = 10; val_ptr[t] = value; continue; }
        value = n_zx_e2.dot(p_zx) + d_zx_e2;
        if (value < 0.0f) { hit_ptr[t] = 0; code_ptr[t] = 11; val_ptr[t] = value; continue; }

        hit_ptr[t] = 1;
        code_ptr[t] = 0;
        val_ptr[t] = 0.0f;
    }
    return std::make_tuple(hit, fail_code, fail_value, normal_trace);
}

torch::Tensor face_qef_cpu_normal_probe(
    const torch::Tensor& triangles,
    int64_t tri_id) {
    const float* tri = triangles.data_ptr<float>() + tri_id * 9;
    const Eigen::Vector3f v0(tri[0], tri[1], tri[2]);
    const Eigen::Vector3f v1(tri[3], tri[4], tri[5]);
    const Eigen::Vector3f v2(tri[6], tri[7], tri[8]);
    const Eigen::Vector3f e0 = v1 - v0;
    const Eigen::Vector3f e1 = v2 - v1;
    const float nx_a = e0.y() * e1.z();
    const float nx_b = e0.z() * e1.y();
    const float ny_a = e0.z() * e1.x();
    const float ny_b = e0.x() * e1.z();
    const float nz_a = e0.x() * e1.y();
    const float nz_b = e0.y() * e1.x();
    const float raw_nx = nx_a - nx_b;
    const float raw_ny = ny_a - ny_b;
    const float raw_nz = nz_a - nz_b;

    auto out = torch::empty({2, 30}, torch::TensorOptions().dtype(torch::kFloat32).device(torch::kCPU));
    float* o = out.data_ptr<float>();
    auto fill = [&](int row, float rx, float ry, float rz, float nx, float ny, float nz) {
        const float n2 = rx * rx + ry * ry + rz * rz;
        const float len = n2 > 0.0f ? static_cast<float>(std::sqrt(n2)) : 0.0f;
        float* r = o + row * 30;
        r[0] = v0.x(); r[1] = v0.y(); r[2] = v0.z();
        r[3] = v1.x(); r[4] = v1.y(); r[5] = v1.z();
        r[6] = v2.x(); r[7] = v2.y(); r[8] = v2.z();
        r[9] = e0.x(); r[10] = e0.y(); r[11] = e0.z();
        r[12] = e1.x(); r[13] = e1.y(); r[14] = e1.z();
        r[15] = nx_a; r[16] = nx_b; r[17] = ny_a; r[18] = ny_b; r[19] = nz_a; r[20] = nz_b;
        r[21] = rx; r[22] = ry; r[23] = rz; r[24] = n2; r[25] = len;
        r[26] = nx; r[27] = ny; r[28] = nz; r[29] = nx * nx + ny * ny + nz * nz;
    };

    float snx = raw_nx;
    float sny = raw_ny;
    float snz = raw_nz;
    const float scalar_n2 = raw_nx * raw_nx + raw_ny * raw_ny + raw_nz * raw_nz;
    if (scalar_n2 > 0.0f) {
        const float len = static_cast<float>(std::sqrt(scalar_n2));
        snx /= len;
        sny /= len;
        snz /= len;
    }
    fill(0, raw_nx, raw_ny, raw_nz, snx, sny, snz);

    const Eigen::Vector3f eigen_raw = e0.cross(e1);
    const Eigen::Vector3f eigen_n = eigen_raw.normalized();
    fill(1, eigen_raw.x(), eigen_raw.y(), eigen_raw.z(), eigen_n.x(), eigen_n.y(), eigen_n.z());
    return out;
}

} // namespace

PYBIND11_MODULE(TORCH_EXTENSION_NAME, m) {
    m.def("intersect_qef", &o_voxel::fdg::intersect_qef);
    m.def("face_qef_old", &o_voxel::fdg::face_qef_old);
    m.def("face_qef_ref", &o_voxel::fdg::face_qef_ref);
    m.def("face_qef", &o_voxel::fdg::face_qef);
    m.def("face_qef_cpu", &face_qef_cpu_only);
    m.def("face_qef_cpu_probe", &face_qef_cpu_probe);
    m.def("face_qef_cpu_normal_probe", &face_qef_cpu_normal_probe);
    m.def("face_qef_gpu_probe", &face_qef_gpu_probe);
    m.def("face_qef_gpu_normal_probe", &face_qef_gpu_normal_probe);
}
""".replace('__ROOT__', ROOT.as_posix())

probe_cu = r"""
#include <torch/extension.h>
#include <ATen/cuda/CUDAContext.h>
#include <c10/cuda/CUDAGuard.h>
#include <c10/cuda/CUDAException.h>
#include <cuda_runtime.h>
#include <tuple>
#include "mesh_to_flexible_dual_grid_gpu/types.cuh"

using o_voxel::fdg::Int3;

namespace {

constexpr int kThreads = 256;

__device__ __forceinline__ float pmin(float a, float b) { return a < b ? a : b; }
__device__ __forceinline__ float pmax(float a, float b) { return a > b ? a : b; }

__device__ __forceinline__ float3 aligned_cross_normalize3(
    float e0x,
    float e0y,
    float e0z,
    float e1x,
    float e1y,
    float e1z) {
    const float nx = __fsub_rn(__fmul_rn(e0y, e1z), __fmul_rn(e0z, e1y));
    const float ny = __fsub_rn(__fmul_rn(e0z, e1x), __fmul_rn(e0x, e1z));
    const float nz = __fsub_rn(__fmul_rn(e0x, e1y), __fmul_rn(e0y, e1x));
    const float n2 = __fadd_rn(
        __fadd_rn(__fmul_rn(nx, nx), __fmul_rn(ny, ny)),
        __fmul_rn(nz, nz));
    if (n2 > 0.0f) {
        const float len = sqrtf(n2);
        return make_float3(nx / len, ny / len, nz / len);
    }
    return make_float3(nx, ny, nz);
}

__global__ void face_qef_probe_kernel(
    int64_t num_triangles,
    const float* __restrict__ triangles,
    float3 voxel_size,
    Int3 grid_min,
    Int3 grid_max,
    int x,
    int y,
    int z,
    uint8_t* __restrict__ hit,
    int32_t* __restrict__ fail_code,
    float* __restrict__ fail_value,
    float* __restrict__ hit_normal_trace,
    float* __restrict__ qef_normal_trace) {
    const int64_t tid = static_cast<int64_t>(blockIdx.x) * blockDim.x + threadIdx.x;
    if (tid >= num_triangles)
        return;

    const float* tri = triangles + tid * 9;
    const float3 v0 = make_float3(tri[0], tri[1], tri[2]);
    const float3 v1 = make_float3(tri[3], tri[4], tri[5]);
    const float3 v2 = make_float3(tri[6], tri[7], tri[8]);
    const float vs[3] = {voxel_size.x, voxel_size.y, voxel_size.z};

    const float e0x = v1.x - v0.x;
    const float e0y = v1.y - v0.y;
    const float e0z = v1.z - v0.z;
    const float e1x = v2.x - v1.x;
    const float e1y = v2.y - v1.y;
    const float e1z = v2.z - v1.z;
    const float e2x = v0.x - v2.x;
    const float e2y = v0.y - v2.y;
    const float e2z = v0.z - v2.z;

    const float3 n = aligned_cross_normalize3(e0x, e0y, e0z, e1x, e1y, e1z);
    const float nx = n.x;
    const float ny = n.y;
    const float nz = n.z;
    const float normal_trace = nx * nx + ny * ny + nz * nz;
    hit_normal_trace[tid] = normal_trace;
    qef_normal_trace[tid] = normal_trace;

    const float bb_min_f_x = pmin(pmin(v0.x, v1.x), v2.x) / vs[0];
    const float bb_min_f_y = pmin(pmin(v0.y, v1.y), v2.y) / vs[1];
    const float bb_min_f_z = pmin(pmin(v0.z, v1.z), v2.z) / vs[2];
    const float bb_max_f_x = pmax(pmax(v0.x, v1.x), v2.x) / vs[0];
    const float bb_max_f_y = pmax(pmax(v0.y, v1.y), v2.y) / vs[1];
    const float bb_max_f_z = pmax(pmax(v0.z, v1.z), v2.z) / vs[2];
    const int bb_min_x = max(static_cast<int>(bb_min_f_x), grid_min.x);
    const int bb_min_y = max(static_cast<int>(bb_min_f_y), grid_min.y);
    const int bb_min_z = max(static_cast<int>(bb_min_f_z), grid_min.z);
    const int bb_max_x = min(static_cast<int>(bb_max_f_x + 1.0f), grid_max.x);
    const int bb_max_y = min(static_cast<int>(bb_max_f_y + 1.0f), grid_max.y);
    const int bb_max_z = min(static_cast<int>(bb_max_f_z + 1.0f), grid_max.z);
    if (x < bb_min_x || x >= bb_max_x || y < bb_min_y || y >= bb_max_y || z < bb_min_z || z >= bb_max_z) {
        hit[tid] = 0;
        fail_code[tid] = 1;
        fail_value[tid] = 0.0f;
        return;
    }

    const float px = x * vs[0];
    const float py = y * vs[1];
    const float pz = z * vs[2];
    const float c_x = nx > 0.0f ? vs[0] : 0.0f;
    const float c_y = ny > 0.0f ? vs[1] : 0.0f;
    const float c_z = nz > 0.0f ? vs[2] : 0.0f;
    const float d1 = nx * (c_x - v0.x) + ny * (c_y - v0.y) + nz * (c_z - v0.z);
    const float d2 = nx * (vs[0] - c_x - v0.x) + ny * (vs[1] - c_y - v0.y) + nz * (vs[2] - c_z - v0.z);
    const float n_dot_p = nx * px + ny * py + nz * pz;
    float value = (n_dot_p + d1) * (n_dot_p + d2);
    if (value > 0.0f) { hit[tid] = 0; fail_code[tid] = 2; fail_value[tid] = value; return; }

    const int mul_xy = nz < 0.0f ? -1 : 1;
    const float n_xy_e0_x = -mul_xy * e0y;
    const float n_xy_e0_y = mul_xy * e0x;
    const float n_xy_e1_x = -mul_xy * e1y;
    const float n_xy_e1_y = mul_xy * e1x;
    const float n_xy_e2_x = -mul_xy * e2y;
    const float n_xy_e2_y = mul_xy * e2x;
    const float d_xy_e0 = -(n_xy_e0_x * v0.x + n_xy_e0_y * v0.y) + fmaxf(n_xy_e0_x, 0.0f) * vs[0] + fmaxf(n_xy_e0_y, 0.0f) * vs[1];
    const float d_xy_e1 = -(n_xy_e1_x * v1.x + n_xy_e1_y * v1.y) + fmaxf(n_xy_e1_x, 0.0f) * vs[0] + fmaxf(n_xy_e1_y, 0.0f) * vs[1];
    const float d_xy_e2 = -(n_xy_e2_x * v2.x + n_xy_e2_y * v2.y) + fmaxf(n_xy_e2_x, 0.0f) * vs[0] + fmaxf(n_xy_e2_y, 0.0f) * vs[1];
    value = n_xy_e0_x * px + n_xy_e0_y * py + d_xy_e0;
    if (value < 0.0f) { hit[tid] = 0; fail_code[tid] = 3; fail_value[tid] = value; return; }
    value = n_xy_e1_x * px + n_xy_e1_y * py + d_xy_e1;
    if (value < 0.0f) { hit[tid] = 0; fail_code[tid] = 4; fail_value[tid] = value; return; }
    value = n_xy_e2_x * px + n_xy_e2_y * py + d_xy_e2;
    if (value < 0.0f) { hit[tid] = 0; fail_code[tid] = 5; fail_value[tid] = value; return; }

    const int mul_yz = nx < 0.0f ? -1 : 1;
    const float n_yz_e0_x = -mul_yz * e0z;
    const float n_yz_e0_y = mul_yz * e0y;
    const float n_yz_e1_x = -mul_yz * e1z;
    const float n_yz_e1_y = mul_yz * e1y;
    const float n_yz_e2_x = -mul_yz * e2z;
    const float n_yz_e2_y = mul_yz * e2y;
    const float d_yz_e0 = -(n_yz_e0_x * v0.y + n_yz_e0_y * v0.z) + fmaxf(n_yz_e0_x, 0.0f) * vs[1] + fmaxf(n_yz_e0_y, 0.0f) * vs[2];
    const float d_yz_e1 = -(n_yz_e1_x * v1.y + n_yz_e1_y * v1.z) + fmaxf(n_yz_e1_x, 0.0f) * vs[1] + fmaxf(n_yz_e1_y, 0.0f) * vs[2];
    const float d_yz_e2 = -(n_yz_e2_x * v2.y + n_yz_e2_y * v2.z) + fmaxf(n_yz_e2_x, 0.0f) * vs[1] + fmaxf(n_yz_e2_y, 0.0f) * vs[2];
    value = n_yz_e0_x * py + n_yz_e0_y * pz + d_yz_e0;
    if (value < 0.0f) { hit[tid] = 0; fail_code[tid] = 6; fail_value[tid] = value; return; }
    value = n_yz_e1_x * py + n_yz_e1_y * pz + d_yz_e1;
    if (value < 0.0f) { hit[tid] = 0; fail_code[tid] = 7; fail_value[tid] = value; return; }
    value = n_yz_e2_x * py + n_yz_e2_y * pz + d_yz_e2;
    if (value < 0.0f) { hit[tid] = 0; fail_code[tid] = 8; fail_value[tid] = value; return; }

    const int mul_zx = ny < 0.0f ? -1 : 1;
    const float n_zx_e0_x = -mul_zx * e0x;
    const float n_zx_e0_y = mul_zx * e0z;
    const float n_zx_e1_x = -mul_zx * e1x;
    const float n_zx_e1_y = mul_zx * e1z;
    const float n_zx_e2_x = -mul_zx * e2x;
    const float n_zx_e2_y = mul_zx * e2z;
    const float d_zx_e0 = -(n_zx_e0_x * v0.z + n_zx_e0_y * v0.x) + fmaxf(n_zx_e0_x, 0.0f) * vs[2] + fmaxf(n_zx_e0_y, 0.0f) * vs[0];
    const float d_zx_e1 = -(n_zx_e1_x * v1.z + n_zx_e1_y * v1.x) + fmaxf(n_zx_e1_x, 0.0f) * vs[2] + fmaxf(n_zx_e1_y, 0.0f) * vs[0];
    const float d_zx_e2 = -(n_zx_e2_x * v2.z + n_zx_e2_y * v2.x) + fmaxf(n_zx_e2_x, 0.0f) * vs[2] + fmaxf(n_zx_e2_y, 0.0f) * vs[0];
    value = n_zx_e0_x * pz + n_zx_e0_y * px + d_zx_e0;
    if (value < 0.0f) { hit[tid] = 0; fail_code[tid] = 9; fail_value[tid] = value; return; }
    value = n_zx_e1_x * pz + n_zx_e1_y * px + d_zx_e1;
    if (value < 0.0f) { hit[tid] = 0; fail_code[tid] = 10; fail_value[tid] = value; return; }
    value = n_zx_e2_x * pz + n_zx_e2_y * px + d_zx_e2;
    if (value < 0.0f) { hit[tid] = 0; fail_code[tid] = 11; fail_value[tid] = value; return; }

    hit[tid] = 1;
    fail_code[tid] = 0;
    fail_value[tid] = 0.0f;
}

__global__ void normal_probe_kernel(
    const float* __restrict__ triangles,
    int64_t tri_id,
    float* __restrict__ out) {
    const float* tri = triangles + tri_id * 9;
    const float v0x = tri[0];
    const float v0y = tri[1];
    const float v0z = tri[2];
    const float v1x = tri[3];
    const float v1y = tri[4];
    const float v1z = tri[5];
    const float v2x = tri[6];
    const float v2y = tri[7];
    const float v2z = tri[8];
    const float e0x = v1x - v0x;
    const float e0y = v1y - v0y;
    const float e0z = v1z - v0z;
    const float e1x = v2x - v1x;
    const float e1y = v2y - v1y;
    const float e1z = v2z - v1z;

    const float nx_a = e0y * e1z;
    const float nx_b = e0z * e1y;
    const float ny_a = e0z * e1x;
    const float ny_b = e0x * e1z;
    const float nz_a = e0x * e1y;
    const float nz_b = e0y * e1x;
    const float raw_nx = e0y * e1z - e0z * e1y;
    const float raw_ny = e0z * e1x - e0x * e1z;
    const float raw_nz = e0x * e1y - e0y * e1x;
    const float n2 = raw_nx * raw_nx + raw_ny * raw_ny + raw_nz * raw_nz;
    const float len = n2 > 0.0f ? sqrtf(n2) : 0.0f;
    const float nnx = n2 > 0.0f ? raw_nx / len : raw_nx;
    const float nny = n2 > 0.0f ? raw_ny / len : raw_ny;
    const float nnz = n2 > 0.0f ? raw_nz / len : raw_nz;
    float* r = out;
    r[0] = v0x; r[1] = v0y; r[2] = v0z;
    r[3] = v1x; r[4] = v1y; r[5] = v1z;
    r[6] = v2x; r[7] = v2y; r[8] = v2z;
    r[9] = e0x; r[10] = e0y; r[11] = e0z;
    r[12] = e1x; r[13] = e1y; r[14] = e1z;
    r[15] = nx_a; r[16] = nx_b; r[17] = ny_a; r[18] = ny_b; r[19] = nz_a; r[20] = nz_b;
    r[21] = raw_nx; r[22] = raw_ny; r[23] = raw_nz; r[24] = n2; r[25] = len;
    r[26] = nnx; r[27] = nny; r[28] = nnz; r[29] = nnx * nnx + nny * nny + nnz * nnz;

    const float rnxa = __fmul_rn(e0y, e1z);
    const float rnxb = __fmul_rn(e0z, e1y);
    const float rnya = __fmul_rn(e0z, e1x);
    const float rnyb = __fmul_rn(e0x, e1z);
    const float rnza = __fmul_rn(e0x, e1y);
    const float rnzb = __fmul_rn(e0y, e1x);
    const float rraw_nx = __fsub_rn(rnxa, rnxb);
    const float rraw_ny = __fsub_rn(rnya, rnyb);
    const float rraw_nz = __fsub_rn(rnza, rnzb);
    const float rn2 = __fadd_rn(__fadd_rn(__fmul_rn(rraw_nx, rraw_nx), __fmul_rn(rraw_ny, rraw_ny)), __fmul_rn(rraw_nz, rraw_nz));
    const float rlen = rn2 > 0.0f ? sqrtf(rn2) : 0.0f;
    const float rnnx = rn2 > 0.0f ? rraw_nx / rlen : rraw_nx;
    const float rnny = rn2 > 0.0f ? rraw_ny / rlen : rraw_ny;
    const float rnnz = rn2 > 0.0f ? rraw_nz / rlen : rraw_nz;
    r = out + 30;
    r[0] = v0x; r[1] = v0y; r[2] = v0z;
    r[3] = v1x; r[4] = v1y; r[5] = v1z;
    r[6] = v2x; r[7] = v2y; r[8] = v2z;
    r[9] = e0x; r[10] = e0y; r[11] = e0z;
    r[12] = e1x; r[13] = e1y; r[14] = e1z;
    r[15] = rnxa; r[16] = rnxb; r[17] = rnya; r[18] = rnyb; r[19] = rnza; r[20] = rnzb;
    r[21] = rraw_nx; r[22] = rraw_ny; r[23] = rraw_nz; r[24] = rn2; r[25] = rlen;
    r[26] = rnnx; r[27] = rnny; r[28] = rnnz; r[29] = rnnx * rnnx + rnny * rnny + rnnz * rnnz;

    const float fnxa = rnxa;
    const float fnxb = rnxb;
    const float fnya = rnya;
    const float fnyb = rnyb;
    const float fnza = rnza;
    const float fnzb = rnzb;
    const float fraw_nx = __fmaf_rn(e0y, e1z, -rnxb);
    const float fraw_ny = __fmaf_rn(e0z, e1x, -rnyb);
    const float fraw_nz = __fmaf_rn(e0x, e1y, -rnzb);
    const float fn2 = fraw_nx * fraw_nx + fraw_ny * fraw_ny + fraw_nz * fraw_nz;
    const float flen = fn2 > 0.0f ? sqrtf(fn2) : 0.0f;
    const float fnnx = fn2 > 0.0f ? fraw_nx / flen : fraw_nx;
    const float fnny = fn2 > 0.0f ? fraw_ny / flen : fraw_ny;
    const float fnnz = fn2 > 0.0f ? fraw_nz / flen : fraw_nz;
    r = out + 60;
    r[0] = v0x; r[1] = v0y; r[2] = v0z;
    r[3] = v1x; r[4] = v1y; r[5] = v1z;
    r[6] = v2x; r[7] = v2y; r[8] = v2z;
    r[9] = e0x; r[10] = e0y; r[11] = e0z;
    r[12] = e1x; r[13] = e1y; r[14] = e1z;
    r[15] = fnxa; r[16] = fnxb; r[17] = fnya; r[18] = fnyb; r[19] = fnza; r[20] = fnzb;
    r[21] = fraw_nx; r[22] = fraw_ny; r[23] = fraw_nz; r[24] = fn2; r[25] = flen;
    r[26] = fnnx; r[27] = fnny; r[28] = fnnz; r[29] = fnnx * fnnx + fnny * fnny + fnnz * fnnz;
}

} // namespace

std::tuple<torch::Tensor, torch::Tensor, torch::Tensor, torch::Tensor, torch::Tensor> face_qef_gpu_probe(
    const torch::Tensor& triangles,
    const torch::Tensor& voxel_size,
    const torch::Tensor& grid_range,
    const torch::Tensor& voxel) {
    const c10::cuda::CUDAGuard guard(triangles.device());
    const cudaStream_t stream = at::cuda::getCurrentCUDAStream(triangles.get_device()).stream();
    const torch::Device device = triangles.device();
    const int64_t T = triangles.size(0);
    const float* vs = voxel_size.data_ptr<float>();
    const int32_t* gr = grid_range.data_ptr<int32_t>();
    const int32_t* vp = voxel.data_ptr<int32_t>();
    const float3 voxel_size_h{vs[0], vs[1], vs[2]};
    const Int3 grid_min{gr[0], gr[1], gr[2]};
    const Int3 grid_max{gr[3], gr[4], gr[5]};
    auto hit = torch::empty({T}, torch::TensorOptions().dtype(torch::kUInt8).device(device));
    auto fail_code = torch::empty({T}, torch::TensorOptions().dtype(torch::kInt32).device(device));
    auto fail_value = torch::empty({T}, torch::TensorOptions().dtype(torch::kFloat32).device(device));
    auto hit_normal_trace = torch::empty({T}, torch::TensorOptions().dtype(torch::kFloat32).device(device));
    auto qef_normal_trace = torch::empty({T}, torch::TensorOptions().dtype(torch::kFloat32).device(device));
    const int blocks = static_cast<int>((T + kThreads - 1) / kThreads);
    face_qef_probe_kernel<<<blocks, kThreads, 0, stream>>>(
        T,
        triangles.data_ptr<float>(),
        voxel_size_h,
        grid_min,
        grid_max,
        vp[0], vp[1], vp[2],
        hit.data_ptr<uint8_t>(),
        fail_code.data_ptr<int32_t>(),
        fail_value.data_ptr<float>(),
        hit_normal_trace.data_ptr<float>(),
        qef_normal_trace.data_ptr<float>());
    C10_CUDA_KERNEL_LAUNCH_CHECK();
    return std::make_tuple(hit, fail_code, fail_value, hit_normal_trace, qef_normal_trace);
}

torch::Tensor face_qef_gpu_normal_probe(
    const torch::Tensor& triangles,
    int64_t tri_id) {
    const c10::cuda::CUDAGuard guard(triangles.device());
    const cudaStream_t stream = at::cuda::getCurrentCUDAStream(triangles.get_device()).stream();
    auto out = torch::empty({3, 30}, torch::TensorOptions().dtype(torch::kFloat32).device(triangles.device()));
    normal_probe_kernel<<<1, 1, 0, stream>>>(
        triangles.data_ptr<float>(),
        tri_id,
        out.data_ptr<float>());
    C10_CUDA_KERNEL_LAUNCH_CHECK();
    return out;
}
""".replace('__ROOT__', ROOT.as_posix())

binding_path.write_text(binding_cpp)
probe_path.write_text(probe_cu)

sources = [
    str(binding_path),
    str(probe_path),
    str(ROOT / 'src/convert/mesh_to_flexible_dual_grid_gpu/intersect_qef.cu'),
    str(ROOT / 'src/convert/mesh_to_flexible_dual_grid_gpu/voxelize_mesh_octree.cu'),
    str(ROOT / 'src/convert/mesh_to_flexible_dual_grid_gpu/face_qef_old.cu'),
    str(ROOT / 'src/convert/mesh_to_flexible_dual_grid_gpu/face_qef_ref.cu'),
    str(ROOT / 'src/convert/mesh_to_flexible_dual_grid_gpu/face_qef.cu'),
]

ext = load(
    name=f'fdg_face_qef_jit_{os.getpid()}',
    sources=sources,
    build_directory=str(torch_build_dir),
    extra_include_paths=[
        str(ROOT / 'src/convert'),
        str(ROOT / 'src/convert/mesh_to_flexible_dual_grid_gpu'),
        str(ROOT / 'third_party/eigen'),
    ],
    extra_cflags=['-O3', '-std=c++17'],
    extra_cuda_cflags=['-O3', '-std=c++17'],
    with_cuda=True,
    verbose=True,
)
print('JIT extension loaded:', ext)


Detected CUDA files, patching ldflags
Emitting ninja build file /tmp/fdg_face_qef_jit_betjkma8/torch_ext/build.ninja...
/home/quanta/.conda/envs/symm-enforce/lib/python3.10/site-packages/torch/utils/cpp_extension.py:2059: UserWarning: TORCH_CUDA_ARCH_LIST is not set, all archs for visible cards are included for compilation. 
If this is not desired, please set os.environ['TORCH_CUDA_ARCH_LIST'].
  warnings.warn(
Building extension module fdg_face_qef_jit_1084134...
Using envvar MAX_JOBS (32) as the number of workers...


[1/8] /home/quanta/.conda/envs/symm-enforce/bin/g++ -MMD -MF binding.o.d -DTORCH_EXTENSION_NAME=fdg_face_qef_jit_1084134 -DTORCH_API_INCLUDE_EXTENSION_H -DPYBIND11_COMPILER_TYPE=\"_gcc\" -DPYBIND11_STDLIB=\"_libstdcpp\" -DPYBIND11_BUILD_ABI=\"_cxxabi1011\" -I/mnt/nvmefs/Projects/o-voxel-gpu/src/convert -I/mnt/nvmefs/Projects/o-voxel-gpu/src/convert/mesh_to_flexible_dual_grid_gpu -I/mnt/nvmefs/Projects/o-voxel-gpu/third_party/eigen -isystem /home/quanta/.conda/envs/symm-enforce/lib/python3.10/site-packages/torch/include -isystem /home/quanta/.conda/envs/symm-enforce/lib/python3.10/site-packages/torch/include/torch/csrc/api/include -isystem /home/quanta/.conda/envs/symm-enforce/lib/python3.10/site-packages/torch/include/TH -isystem /home/quanta/.conda/envs/symm-enforce/lib/python3.10/site-packages/torch/include/THC -isystem /home/quanta/.conda/envs/symm-enforce/include -isystem /home/quanta/.conda/envs/symm-enforce/include/python3.10 -D_GLIBCXX_USE_CXX11_ABI=0 -fPIC -std=c++17 -O3 -std=c

Loading extension module fdg_face_qef_jit_1084134...


## Load Mesh Inputs

The original mesh comes from `notebooks/test.glb`. The subdivided input uses the same GPU subdivision helper as the intersect notebook.

In [3]:
import numpy as np
import trimesh


def subdivide_mesh_gpu(verts: torch.Tensor, faces: torch.Tensor, area_threshold: float, max_iters: int):
    from pytorch3d.structures import Meshes

    device = verts.device
    cur = Meshes(verts=[verts], faces=[faces])

    for _ in range(max_iters):
        cv = cur.verts_packed()
        cf = cur.faces_packed()
        edges = cur.edges_packed()
        f2e = cur.faces_packed_to_edges_packed()

        areas = cur.faces_areas_packed()
        large = areas > area_threshold
        if not large.any():
            break

        edge_len = torch.norm(cv[edges[:, 0]] - cv[edges[:, 1]], dim=1)
        _, local_max = torch.max(edge_len[f2e], dim=1)
        max_edge_ids = f2e[torch.arange(cf.shape[0], device=device), local_max]

        target = torch.zeros(edges.shape[0], dtype=torch.bool, device=device)
        target[max_edge_ids[large]] = True
        if not target.any():
            break

        mid = (cv[edges[target, 0]] + cv[edges[target, 1]]) * 0.5
        new_verts = torch.cat([cv, mid], dim=0)
        n_old = cv.shape[0]

        e2new = torch.zeros(edges.shape[0], dtype=torch.long, device=device)
        e2new[target] = torch.arange(n_old, n_old + target.sum().item(), device=device)

        split_counts = target[f2e].sum(dim=1)

        v0, v1, v2 = cf[:, 0], cf[:, 1], cf[:, 2]
        p01_a = torch.minimum(v0, v1)
        p01_b = torch.maximum(v0, v1)
        p12_a = torch.minimum(v1, v2)
        p12_b = torch.maximum(v1, v2)
        p20_a = torch.minimum(v2, v0)
        p20_b = torch.maximum(v2, v0)

        e_act = edges[f2e]
        match01 = (e_act[:, :, 0] == p01_a.unsqueeze(1)) & (e_act[:, :, 1] == p01_b.unsqueeze(1))
        match12 = (e_act[:, :, 0] == p12_a.unsqueeze(1)) & (e_act[:, :, 1] == p12_b.unsqueeze(1))
        match20 = (e_act[:, :, 0] == p20_a.unsqueeze(1)) & (e_act[:, :, 1] == p20_b.unsqueeze(1))

        col01 = match01.long().argmax(dim=1)
        col12 = match12.long().argmax(dim=1)
        col20 = match20.long().argmax(dim=1)

        keep = cf[split_counts == 0]
        out = [keep]

        m1 = split_counts == 1
        if m1.any():
            f1 = cf[m1]
            fe1 = f2e[m1]
            M = f1.shape[0]
            v0_1, v1_1, v2_1 = f1[:, 0], f1[:, 1], f1[:, 2]

            split_col = target[fe1].long().argmax(dim=1)
            c01_m1 = col01[m1]
            c12_m1 = col12[m1]
            c20_m1 = col20[m1]
            is_split_01 = split_col == c01_m1
            is_split_12 = split_col == c12_m1
            is_split_20 = split_col == c20_m1

            split_eid = fe1[torch.arange(M, device=device), split_col]
            vn = e2new[split_eid]

            s1 = torch.zeros(M, 3, dtype=torch.long, device=device)
            s2 = torch.zeros(M, 3, dtype=torch.long, device=device)

            s1[is_split_01] = torch.stack([v0_1[is_split_01], vn[is_split_01], v2_1[is_split_01]], dim=1)
            s2[is_split_01] = torch.stack([vn[is_split_01], v1_1[is_split_01], v2_1[is_split_01]], dim=1)
            s1[is_split_12] = torch.stack([v0_1[is_split_12], v1_1[is_split_12], vn[is_split_12]], dim=1)
            s2[is_split_12] = torch.stack([v0_1[is_split_12], vn[is_split_12], v2_1[is_split_12]], dim=1)
            s1[is_split_20] = torch.stack([v0_1[is_split_20], v1_1[is_split_20], vn[is_split_20]], dim=1)
            s2[is_split_20] = torch.stack([vn[is_split_20], v1_1[is_split_20], v2_1[is_split_20]], dim=1)

            out.extend([s1, s2])

        mm = split_counts >= 2
        if mm.any():
            fm = cf[mm]
            fem = f2e[mm]
            M2 = fm.shape[0]
            v0_m, v1_m, v2_m = fm[:, 0], fm[:, 1], fm[:, 2]

            c01_m = col01[mm]
            c12_m = col12[mm]
            c20_m = col20[mm]

            eid01 = fem[torch.arange(M2, device=device), c01_m]
            e01_is_split = target[eid01]
            e01_v = torch.where(e01_is_split, e2new[eid01], v0_m)

            eid12 = fem[torch.arange(M2, device=device), c12_m]
            e12_is_split = target[eid12]
            e12_v = torch.where(e12_is_split, e2new[eid12], v1_m)

            eid20 = fem[torch.arange(M2, device=device), c20_m]
            e20_is_split = target[eid20]
            e20_v = torch.where(e20_is_split, e2new[eid20], v2_m)

            mf1 = torch.stack([v0_m, e01_v, e20_v], dim=1)
            mf2 = torch.stack([e01_v, v1_m, e12_v], dim=1)
            mf3 = torch.stack([e20_v, e12_v, v2_m], dim=1)
            mf4 = torch.stack([e01_v, e12_v, e20_v], dim=1)
            out.extend([mf1, mf2, mf3, mf4])

        cur = Meshes(verts=[new_verts], faces=[torch.cat(out, dim=0)])

    return cur.verts_packed(), cur.faces_packed()


def triangle_mb(triangles: torch.Tensor) -> float:
    return triangles.numel() * triangles.element_size() / 1024**2


glb_path = ROOT / 'notebooks/test.glb'
loaded = trimesh.load(glb_path, force='scene')
if isinstance(loaded, trimesh.Scene):
    meshes = [g for g in loaded.geometry.values() if isinstance(g, trimesh.Trimesh) and len(g.faces) > 0]
    mesh = trimesh.util.concatenate(meshes)
else:
    mesh = loaded
assert isinstance(mesh, trimesh.Trimesh) and len(mesh.faces) > 0

vertices_np = np.asarray(mesh.vertices, dtype=np.float32)
faces_np = np.asarray(mesh.faces, dtype=np.int64)
vmin = vertices_np.min(axis=0)
vmax = vertices_np.max(axis=0)
center = (vmin + vmax) * 0.5
scale = np.float32(0.999 / max((vmax - vmin).max(), 1e-12))
vertices_np = (vertices_np - center) * scale + np.float32(0.5)

vertices_cpu = torch.from_numpy(vertices_np).contiguous()
faces_cpu = torch.from_numpy(faces_np).contiguous()
triangles_cpu = vertices_cpu[faces_cpu].reshape(-1, 9).contiguous()
triangles_cuda = triangles_cpu.to(DEVICE, non_blocking=True).contiguous()

voxel_size_cpu = torch.tensor([1.0 / GRID_SIZE, 1.0 / GRID_SIZE, 1.0 / GRID_SIZE], dtype=torch.float32)
grid_range_cpu = torch.tensor([0, 0, 0, GRID_SIZE, GRID_SIZE, GRID_SIZE], dtype=torch.int32)

print('building subdivided input...')
torch.cuda.empty_cache()
torch.cuda.synchronize()
start = time.perf_counter()
sub_vertices_cuda, sub_faces_cuda = subdivide_mesh_gpu(
    vertices_cpu.to(DEVICE, non_blocking=True).contiguous(),
    faces_cpu.to(DEVICE, non_blocking=True).contiguous(),
    area_threshold=SUBDIVIDE_AREA_THRESHOLD,
    max_iters=SUBDIVIDE_MAX_ITERS,
)
sub_triangles_cuda = sub_vertices_cuda[sub_faces_cuda].reshape(-1, 9).contiguous()
torch.cuda.synchronize()
subdivide_ms = (time.perf_counter() - start) * 1000.0
sub_triangles_cpu = sub_triangles_cuda.detach().cpu().contiguous()
sub_vertices_shape = tuple(sub_vertices_cuda.shape)
sub_faces_shape = tuple(sub_faces_cuda.shape)
del sub_vertices_cuda, sub_faces_cuda
torch.cuda.empty_cache()

INPUT_CASES = {
    'original': {
        'triangles_cpu': triangles_cpu,
        'triangles_cuda': triangles_cuda,
        'vertices_shape': tuple(vertices_cpu.shape),
        'faces_shape': tuple(faces_cpu.shape),
        'triangles': int(triangles_cpu.shape[0]),
        'triangle_mb': triangle_mb(triangles_cuda),
        'preprocess_ms': 0.0,
    },
    'subdivided': {
        'triangles_cpu': sub_triangles_cpu,
        'triangles_cuda': sub_triangles_cuda,
        'vertices_shape': sub_vertices_shape,
        'faces_shape': sub_faces_shape,
        'triangles': int(sub_triangles_cpu.shape[0]),
        'triangle_mb': triangle_mb(sub_triangles_cuda),
        'preprocess_ms': subdivide_ms,
    },
}

for name, case in INPUT_CASES.items():
    print(
        name,
        'vertices:', case['vertices_shape'],
        'faces:', case['faces_shape'],
        'triangles:', case['triangles'],
        'triangle_mb:', case['triangle_mb'],
        'preprocess_ms:', case['preprocess_ms'],
    )
print('face multiplier:', float(INPUT_CASES['subdivided']['triangles'] / max(INPUT_CASES['original']['triangles'], 1)))
print('voxel_size:', voxel_size_cpu.tolist(), 'grid_range:', grid_range_cpu.tolist())


building subdivided input...
original vertices: (268018, 3) faces: (280333, 3) triangles: 280333 triangle_mb: 9.624469757080078 preprocess_ms: 0.0
subdivided vertices: (3608864, 3) faces: (10304077, 3) triangles: 10304077 triangle_mb: 353.7624092102051 preprocess_ms: 781.4290480018826
face multiplier: 36.75656094715927
voxel_size: [0.001953125, 0.001953125, 0.001953125] grid_range: [0, 0, 0, 512, 512, 512]


## Helpers

GPU functions take CUDA triangles. `voxel_size` and `grid_range` stay as CPU tensors because the C++ functions read their scalar values on host.

In [4]:
from typing import NamedTuple


class IntersectOutput(NamedTuple):
    voxels: torch.Tensor
    mean_sum: torch.Tensor
    cnt: torch.Tensor
    intersected: torch.Tensor
    qefs: torch.Tensor
    brick_hash_keys: torch.Tensor
    brick_hash_vals: torch.Tensor
    brick_bits: torch.Tensor
    brick_base: torch.Tensor


def as_intersect_output(output):
    assert len(output) == 9, f'expected 9 tensors from intersect_qef, got {len(output)}'
    return IntersectOutput(*output)


def voxel_key(voxels: torch.Tensor) -> torch.Tensor:
    voxels = voxels.to(torch.int64)
    rel = voxels - grid_range_cpu[:3].to(torch.int64)
    size = (grid_range_cpu[3:] - grid_range_cpu[:3]).to(torch.int64)
    return rel[:, 0] + size[0] * (rel[:, 1] + size[1] * rel[:, 2])


def canonical_face_qef(voxels: torch.Tensor, qefs: torch.Tensor):
    voxels_cpu = voxels.detach().cpu().contiguous()
    qefs_cpu = qefs.detach().cpu().float().contiguous()
    keys = voxel_key(voxels_cpu)
    order = torch.argsort(keys)
    keys = keys[order]
    unique = torch.unique_consecutive(keys).numel() == keys.numel()
    return {
        'keys': keys,
        'voxels': voxels_cpu[order],
        'qefs': qefs_cpu[order],
        'unique': bool(unique),
    }


def compare_face_qef(cpu, other):
    row = {
        'count_cpu': int(cpu['keys'].numel()),
        'count_other': int(other['keys'].numel()),
        'unique_other': other['unique'],
    }
    if not torch.equal(cpu['keys'], other['keys']):
        a = set(cpu['keys'].tolist())
        b = set(other['keys'].tolist())
        row.update({
            'voxel_keys_equal': False,
            'missing': len(a - b),
            'extra': len(b - a),
            'missing_sample': sorted(list(a - b))[:8],
            'extra_sample': sorted(list(b - a))[:8],
        })
        return row
    diff = (other['qefs'] - cpu['qefs']).float()
    ref = cpu['qefs'].float()
    row.update({
        'voxel_keys_equal': True,
        'missing': 0,
        'extra': 0,
        'qef_max_abs': float(diff.abs().max().item()) if diff.numel() else 0.0,
        'qef_rmse': float(torch.sqrt(torch.mean(diff * diff)).item()) if diff.numel() else 0.0,
        'qef_rel_l2': float(torch.linalg.vector_norm(diff).item() / max(torch.linalg.vector_norm(ref).item(), 1e-12)) if diff.numel() else 0.0,
        'qef_cpu_max_abs': float(ref.abs().max().item()) if ref.numel() else 0.0,
        'qef_other_max_abs': float(other['qefs'].abs().max().item()) if other['qefs'].numel() else 0.0,
    })
    return row


def show_rows(rows):
    try:
        import pandas as pd
        display(pd.DataFrame(rows))
    except Exception:
        for row in rows:
            print(row)


def face_old(case, inter):
    return ext.face_qef_old(case['triangles_cuda'], voxel_size_cpu, grid_range_cpu, inter.voxels)


def face_ref(case, inter):
    return ext.face_qef_ref(
        case['triangles_cuda'], voxel_size_cpu, grid_range_cpu,
        inter.voxels, inter.brick_hash_keys, inter.brick_hash_vals, inter.brick_bits, inter.brick_base)


def face_new(case, inter):
    return ext.face_qef(
        case['triangles_cuda'], voxel_size_cpu, grid_range_cpu,
        inter.voxels, inter.brick_hash_keys, inter.brick_hash_vals, inter.brick_bits, inter.brick_base)


def face_cpu(case, inter):
    return ext.face_qef_cpu(case['triangles_cpu'], voxel_size_cpu, grid_range_cpu, inter.voxels.detach().cpu().contiguous())


## Build Active Voxel Inputs

This cell runs only the current `intersect_qef`. Its output supplies the shared `voxels` and brick lookup tensors for all face QEF measurements.

In [5]:
torch.cuda.empty_cache()
torch.cuda.synchronize()

INTERSECT_OUTPUTS = {}
intersect_rows = []
for input_name, case in INPUT_CASES.items():
    torch.cuda.synchronize()
    torch.cuda.reset_peak_memory_stats()
    base_alloc = torch.cuda.memory_allocated()
    base_reserved = torch.cuda.memory_reserved()
    start = torch.cuda.Event(enable_timing=True)
    end = torch.cuda.Event(enable_timing=True)
    start.record()
    out = as_intersect_output(ext.intersect_qef(case['triangles_cuda'], voxel_size_cpu, grid_range_cpu, CHUNK_TRIANGLES))
    end.record()
    torch.cuda.synchronize()
    INTERSECT_OUTPUTS[input_name] = out
    intersect_rows.append({
        'input': input_name,
        'triangles': case['triangles'],
        'voxels': int(out.voxels.shape[0]),
        'brick_hash_capacity': int(out.brick_hash_keys.shape[0]),
        'brick_slots': int(out.brick_bits.shape[0]),
        'intersect_ms': float(start.elapsed_time(end)),
        'intersect_peak_alloc_mb': float((torch.cuda.max_memory_allocated() - base_alloc) / 1024**2),
        'intersect_peak_reserved_mb': float((torch.cuda.max_memory_reserved() - base_reserved) / 1024**2),
    })

show_rows(intersect_rows)


,input,triangles,voxels,brick_hash_capacity,brick_slots,intersect_ms,intersect_peak_alloc_mb,intersect_peak_reserved_mb
0,original,280333,4676307,524288,262144,26.157248,382.233398,340.0
1,subdivided,10304077,4676306,524288,262144,50.848766,1227.611816,1202.0


## Correctness Against CPU face_qef

All outputs are canonicalized by voxel key before error metrics, so row-order differences do not affect evaluation.

In [6]:
FACE_CANON = {}
correctness_rows = []

for input_name, case in INPUT_CASES.items():
    inter = INTERSECT_OUTPUTS[input_name]
    cpu_allowed = input_name != 'subdivided' or RUN_CPU_SUBDIVIDED
    if not cpu_allowed:
        print(f'skip CPU face_qef for {input_name}; set FDG_RUN_CPU_SUBDIVIDED=1 to enable')
        continue

    print('running CPU face_qef for', input_name, 'triangles:', case['triangles'], 'voxels:', int(inter.voxels.shape[0]))
    cpu_qef = face_cpu(case, inter)
    cpu_can = canonical_face_qef(inter.voxels, cpu_qef)
    FACE_CANON[(input_name, 'cpu')] = cpu_can
    assert cpu_can['unique'], f'{input_name} CPU voxel keys are not unique'

    gpu_outputs = {
        'old': face_old(case, inter),
        'ref': face_ref(case, inter),
        'new': face_new(case, inter),
    }
    torch.cuda.synchronize()

    for method, qef in gpu_outputs.items():
        can = canonical_face_qef(inter.voxels, qef)
        FACE_CANON[(input_name, method)] = can
        row = {'input': input_name, 'method': method, **compare_face_qef(cpu_can, can)}
        correctness_rows.append(row)
        assert row['voxel_keys_equal'], f'{input_name} {method} voxel keys differ from CPU'

show_rows(correctness_rows)


running CPU face_qef for original triangles: 280333 voxels: 4676307
running CPU face_qef for subdivided triangles: 10304077 voxels: 4676306


,input,method,count_cpu,count_other,unique_other,voxel_keys_equal,missing,extra,qef_max_abs,qef_rmse,qef_rel_l2,qef_cpu_max_abs,qef_other_max_abs
0,original,old,4676307,4676307,True,True,0,0,4.360647,0.001880,0.002370,52.151131,52.151131
1,original,ref,4676307,4676307,True,True,0,0,1.331235,0.001508,0.001901,52.151131,52.151127
2,original,new,4676307,4676307,True,True,0,0,1.331235,0.001533,0.001933,52.151131,52.151134
3,subdivided,old,4676306,4676306,True,True,0,0,5.954421,0.007169,0.002442,98.073151,98.073166
4,subdivided,ref,4676306,4676306,True,True,0,0,2.243969,0.003140,0.001069,98.073151,98.073158
5,subdivided,new,4676306,4676306,True,True,0,0,2.243969,0.003070,0.001046,98.073151,98.073158


## QEF Trace Diagnostic

`q00 + q11 + q22` is approximately the number of unit-normal face planes accumulated per voxel. Large trace deltas indicate hit-count differences rather than only plane-coefficient noise.

In [7]:
trace_rows = []
trace_samples = []
for input_name in INPUT_CASES.keys():
    cpu_key = (input_name, 'cpu')
    if cpu_key not in FACE_CANON:
        continue
    cpu = FACE_CANON[cpu_key]
    cpu_trace = cpu['qefs'][:, 0] + cpu['qefs'][:, 4] + cpu['qefs'][:, 7]
    for method in ['old', 'ref', 'new']:
        key = (input_name, method)
        if key not in FACE_CANON:
            continue
        other = FACE_CANON[key]
        other_trace = other['qefs'][:, 0] + other['qefs'][:, 4] + other['qefs'][:, 7]
        d = (other_trace - cpu_trace).float()
        abs_d = d.abs()
        pos = d[d > 0]
        neg = d[d < 0]
        trace_rows.append({
            'input': input_name,
            'method': method,
            'max_abs': float(abs_d.max().item()) if abs_d.numel() else 0.0,
            'mean_abs': float(abs_d.mean().item()) if abs_d.numel() else 0.0,
            'rmse': float(torch.sqrt(torch.mean(d * d)).item()) if d.numel() else 0.0,
            'p50_abs': float(torch.quantile(abs_d, 0.50).item()) if abs_d.numel() else 0.0,
            'p90_abs': float(torch.quantile(abs_d, 0.90).item()) if abs_d.numel() else 0.0,
            'p99_abs': float(torch.quantile(abs_d, 0.99).item()) if abs_d.numel() else 0.0,
            'p999_abs': float(torch.quantile(abs_d, 0.999).item()) if abs_d.numel() else 0.0,
            'positive_sum': float(pos.sum().item()) if pos.numel() else 0.0,
            'negative_sum': float(neg.sum().item()) if neg.numel() else 0.0,
            'positive_voxels': int(pos.numel()),
            'negative_voxels': int(neg.numel()),
        })
        k = min(8, int(abs_d.numel()))
        if k:
            vals, idx = torch.topk(abs_d, k)
            for rank, row_id in enumerate(idx.tolist()):
                trace_samples.append({
                    'input': input_name,
                    'method': method,
                    'rank': rank,
                    'voxel_key': int(cpu['keys'][row_id].item()),
                    'voxel': tuple(int(v) for v in cpu['voxels'][row_id].tolist()),
                    'cpu_trace': float(cpu_trace[row_id].item()),
                    'gpu_trace': float(other_trace[row_id].item()),
                    'trace_diff': float(d[row_id].item()),
                    'abs_trace_diff': float(vals[rank].item()),
                })

show_rows(trace_rows)
show_rows(trace_samples)


,input,method,max_abs,mean_abs,rmse,p50_abs,p90_abs,p99_abs,p999_abs,positive_sum,negative_sum,positive_voxels,negative_voxels
0,original,old,5.000000,0.000017,0.004894,0.0,0.000000e+00,4.768372e-07,9.536743e-07,42.046173,-36.046001,142260,142621
1,original,ref,1.000000,0.000015,0.003924,0.0,0.000000e+00,4.768372e-07,9.536743e-07,33.047226,-39.046772,143912,144476
2,original,new,1.000002,0.000016,0.003951,0.0,0.000000e+00,4.768372e-07,9.536743e-07,41.040779,-32.040546,129243,130216
3,subdivided,old,7.000000,0.000179,0.018716,0.0,9.536743e-07,1.907349e-06,3.814697e-06,644.412659,-194.414108,533516,523561
4,subdivided,ref,2.000001,0.000068,0.008298,0.0,0.000000e+00,9.536743e-07,3.814697e-06,124.142204,-194.140152,169274,167220
5,subdivided,new,2.000001,0.000066,0.008168,0.0,0.000000e+00,9.536743e-07,1.907349e-06,120.106125,-188.104279,134122,132619


,input,method,rank,voxel_key,voxel,cpu_trace,gpu_trace,trace_diff,abs_trace_diff
0,original,old,0,84998193,"(49, 124, 324)",4.000000,9.000000,5.000000,5.000000
1,original,old,1,83614954,"(234, 494, 318)",4.000000,7.000000,3.000000,3.000000
2,original,old,2,62071377,"(81, 401, 236)",8.000000,11.000000,3.000000,3.000000
3,original,old,3,67397978,"(346, 52, 257)",8.000000,10.000000,2.000000,2.000000
4,original,old,4,65742739,"(403, 403, 250)",14.000001,13.000000,-1.000001,1.000001
5,original,old,5,63508884,"(404, 136, 242)",5.000000,6.000000,1.000000,1.000000
6,original,old,6,78727610,"(442, 164, 300)",6.000000,5.000000,-1.000000,1.000000
7,original,old,7,67579681,"(289, 407, 257)",3.000000,2.000000,-1.000000,1.000000
8,original,ref,0,78727610,"(442, 164, 300)",6.000000,5.000000,-1.000000,1.000000
9,original,ref,1,69534791,"(71, 130, 265)",6.000000,5.000000,-1.000000,1.000000


## CPU/GPU Hit Predicate Probe

This probes one voxel from the trace table and compares CPU vs GPU per-triangle hit decisions. It is intended to identify which predicate admits GPU-only hits.

In [8]:
FAIL_NAMES = {
    0: 'hit',
    1: 'bbox',
    2: 'plane',
    3: 'xy_e0',
    4: 'xy_e1',
    5: 'xy_e2',
    6: 'yz_e0',
    7: 'yz_e1',
    8: 'yz_e2',
    9: 'zx_e0',
    10: 'zx_e1',
    11: 'zx_e2',
}

PROBE_INPUT = os.environ.get('FDG_PROBE_INPUT', 'subdivided')
PROBE_METHOD = os.environ.get('FDG_PROBE_METHOD', 'new')
PROBE_RANK = int(os.environ.get('FDG_PROBE_RANK', '0'))
probe_match = [
    r for r in trace_samples
    if r['input'] == PROBE_INPUT and r['method'] == PROBE_METHOD and r['rank'] == PROBE_RANK
]
assert probe_match, (PROBE_INPUT, PROBE_METHOD, PROBE_RANK)
probe_voxel_tuple = tuple(int(v) for v in probe_match[0]['voxel'])
probe_voxel = torch.tensor(probe_voxel_tuple, dtype=torch.int32)
case = INPUT_CASES[PROBE_INPUT]
print('probe target:', {'input': PROBE_INPUT, 'method': PROBE_METHOD, 'rank': PROBE_RANK, 'voxel': probe_voxel_tuple})

cpu_hit_u8, cpu_fail, cpu_val, cpu_normal_trace = ext.face_qef_cpu_probe(
    case['triangles_cpu'], voxel_size_cpu, grid_range_cpu, probe_voxel)
gpu_hit_u8, gpu_fail, gpu_val, gpu_hit_normal_trace, gpu_qef_normal_trace = ext.face_qef_gpu_probe(
    case['triangles_cuda'], voxel_size_cpu, grid_range_cpu, probe_voxel)
torch.cuda.synchronize()

gpu_hit_u8 = gpu_hit_u8.detach().cpu()
gpu_fail = gpu_fail.detach().cpu()
gpu_val = gpu_val.detach().cpu()
gpu_hit_normal_trace = gpu_hit_normal_trace.detach().cpu()
gpu_qef_normal_trace = gpu_qef_normal_trace.detach().cpu()

cpu_hit = cpu_hit_u8.bool()
gpu_hit = gpu_hit_u8.bool()
gpu_extra = (~cpu_hit) & gpu_hit
gpu_missing = cpu_hit & (~gpu_hit)
both_hit = cpu_hit & gpu_hit
both_miss = (~cpu_hit) & (~gpu_hit)

probe_summary = [{
    'input': PROBE_INPUT,
    'voxel': probe_voxel_tuple,
    'triangles': int(cpu_hit.numel()),
    'cpu_hits': int(cpu_hit.sum().item()),
    'gpu_hits': int(gpu_hit.sum().item()),
    'both_hit': int(both_hit.sum().item()),
    'both_miss': int(both_miss.sum().item()),
    'gpu_extra_hits': int(gpu_extra.sum().item()),
    'gpu_missing_hits': int(gpu_missing.sum().item()),
    'cpu_trace_sum': float(cpu_normal_trace[cpu_hit].sum().item()),
    'gpu_hit_trace_sum': float(gpu_hit_normal_trace[gpu_hit].sum().item()),
    'gpu_qef_trace_sum': float(gpu_qef_normal_trace[gpu_hit].sum().item()),
}]
show_rows(probe_summary)


def fail_stats(label, mask, fail, value):
    rows = []
    idx = torch.nonzero(mask, as_tuple=False).flatten()
    if idx.numel() == 0:
        return rows
    codes, counts = torch.unique(fail[idx], return_counts=True)
    for code, count in zip(codes.tolist(), counts.tolist()):
        vals = value[idx[fail[idx] == code]].float()
        abs_vals = vals.abs()
        rows.append({
            'label': label,
            'fail_code': int(code),
            'fail_name': FAIL_NAMES.get(int(code), str(int(code))),
            'count': int(count),
            'value_min': float(vals.min().item()),
            'value_max': float(vals.max().item()),
            'abs_min': float(abs_vals.min().item()),
            'abs_p50': float(torch.quantile(abs_vals, 0.50).item()),
            'abs_p90': float(torch.quantile(abs_vals, 0.90).item()),
            'abs_p99': float(torch.quantile(abs_vals, 0.99).item()),
            'abs_max': float(abs_vals.max().item()),
        })
    return rows

stats_rows = []
stats_rows += fail_stats('gpu_extra_cpu_reject_reason', gpu_extra, cpu_fail, cpu_val)
stats_rows += fail_stats('gpu_missing_gpu_reject_reason', gpu_missing, gpu_fail, gpu_val)
show_rows(stats_rows)


def sample_rows(label, mask, value_for_sort, k=20):
    idx = torch.nonzero(mask, as_tuple=False).flatten()
    if idx.numel() == 0:
        return []
    order = torch.argsort(value_for_sort[idx].abs())[:min(k, int(idx.numel()))]
    tri_ids = idx[order]
    rows = []
    for rank, tri_id_t in enumerate(tri_ids.tolist()):
        tri_id = int(tri_id_t)
        rows.append({
            'label': label,
            'rank': rank,
            'tri_id': tri_id,
            'cpu_hit': bool(cpu_hit[tri_id].item()),
            'gpu_hit': bool(gpu_hit[tri_id].item()),
            'cpu_fail': FAIL_NAMES.get(int(cpu_fail[tri_id].item()), str(int(cpu_fail[tri_id].item()))),
            'gpu_fail': FAIL_NAMES.get(int(gpu_fail[tri_id].item()), str(int(gpu_fail[tri_id].item()))),
            'cpu_fail_value': float(cpu_val[tri_id].item()),
            'gpu_fail_value': float(gpu_val[tri_id].item()),
            'cpu_normal_trace': float(cpu_normal_trace[tri_id].item()),
            'gpu_hit_normal_trace': float(gpu_hit_normal_trace[tri_id].item()),
            'gpu_qef_normal_trace': float(gpu_qef_normal_trace[tri_id].item()),
        })
    return rows

sample = []
sample += sample_rows('gpu_extra_nearest_cpu_boundary', gpu_extra, cpu_val)
sample += sample_rows('gpu_missing_nearest_gpu_boundary', gpu_missing, gpu_val)
show_rows(sample)

normal_gap = gpu_qef_normal_trace - cpu_normal_trace
suspect = both_hit & (cpu_normal_trace.abs() < 1e-6) & (gpu_qef_normal_trace > 0.5)
if suspect.any():
    suspect_idx = torch.nonzero(suspect, as_tuple=False).flatten()
    tri_id = int(suspect_idx[torch.argmax(normal_gap[suspect_idx]).item()].item())
    suspect_reason = 'both_hit_cpu_zero_gpu_unit_normal'
else:
    suspect_idx = torch.nonzero(both_hit, as_tuple=False).flatten()
    tri_id = int(suspect_idx[torch.argmax(normal_gap[suspect_idx].abs()).item()].item())
    suspect_reason = 'largest_both_hit_normal_gap'

cpu_normal_detail = ext.face_qef_cpu_normal_probe(case['triangles_cpu'], tri_id).float()
gpu_normal_detail = ext.face_qef_gpu_normal_probe(case['triangles_cuda'], tri_id).detach().cpu().float()
torch.cuda.synchronize()

normal_fields = [
    'v0.x', 'v0.y', 'v0.z', 'v1.x', 'v1.y', 'v1.z', 'v2.x', 'v2.y', 'v2.z',
    'e0.x', 'e0.y', 'e0.z', 'e1.x', 'e1.y', 'e1.z',
    'nx_a=e0.y*e1.z', 'nx_b=e0.z*e1.y', 'ny_a=e0.z*e1.x', 'ny_b=e0.x*e1.z', 'nz_a=e0.x*e1.y', 'nz_b=e0.y*e1.x',
    'raw_nx', 'raw_ny', 'raw_nz', 'n2', 'len', 'n.x', 'n.y', 'n.z', 'trace',
]
normal_probe_summary = [{
    'tri_id': tri_id,
    'reason': suspect_reason,
    'suspect_count': int(suspect.sum().item()),
    'cpu_hit': bool(cpu_hit[tri_id].item()),
    'gpu_hit': bool(gpu_hit[tri_id].item()),
    'cpu_normal_trace': float(cpu_normal_trace[tri_id].item()),
    'gpu_hit_normal_trace': float(gpu_hit_normal_trace[tri_id].item()),
    'gpu_qef_normal_trace': float(gpu_qef_normal_trace[tri_id].item()),
}]
show_rows(normal_probe_summary)

normal_rows = []
for i, field in enumerate(normal_fields):
    cpu_scalar = float(cpu_normal_detail[0, i].item())
    cpu_eigen = float(cpu_normal_detail[1, i].item())
    gpu_default = float(gpu_normal_detail[0, i].item())
    gpu_no_fma = float(gpu_normal_detail[1, i].item())
    gpu_fma = float(gpu_normal_detail[2, i].item())
    normal_rows.append({
        'field': field,
        'cpu_scalar': cpu_scalar,
        'cpu_eigen': cpu_eigen,
        'gpu_default': gpu_default,
        'gpu_no_fma': gpu_no_fma,
        'gpu_fma': gpu_fma,
        'default_minus_cpu_eigen': gpu_default - cpu_eigen,
        'no_fma_minus_cpu_eigen': gpu_no_fma - cpu_eigen,
        'fma_minus_cpu_eigen': gpu_fma - cpu_eigen,
    })
show_rows(normal_rows)


probe target: {'input': 'subdivided', 'method': 'new', 'rank': 0, 'voxel': (397, 373, 269)}


,input,voxel,triangles,cpu_hits,gpu_hits,both_hit,both_miss,gpu_extra_hits,gpu_missing_hits,cpu_trace_sum,gpu_hit_trace_sum,gpu_qef_trace_sum
0,subdivided,"(397, 373, 269)",10304077,11,8,8,10304066,0,3,8.0,6.0,6.0


,label,fail_code,fail_name,count,value_min,value_max,abs_min,abs_p50,abs_p90,abs_p99,abs_max
0,gpu_missing_gpu_reject_reason,2,plane,2,3.280292e-10,3.280292e-10,3.280292e-10,3.280292e-10,3.280292e-10,3.280292e-10,3.280292e-10
1,gpu_missing_gpu_reject_reason,4,xy_e1,1,-2.328306e-10,-2.328306e-10,2.328306e-10,2.328306e-10,2.328306e-10,2.328306e-10,2.328306e-10


,label,rank,tri_id,cpu_hit,gpu_hit,cpu_fail,gpu_fail,cpu_fail_value,gpu_fail_value,cpu_normal_trace,gpu_hit_normal_trace,gpu_qef_normal_trace
0,gpu_missing_nearest_gpu_boundary,0,1980041,True,False,hit,xy_e1,0.0,-2.328306e-10,0.0,0.0,0.0
1,gpu_missing_nearest_gpu_boundary,1,1679560,True,False,hit,plane,0.0,3.280292e-10,1.0,1.0,1.0
2,gpu_missing_nearest_gpu_boundary,2,1998644,True,False,hit,plane,0.0,3.280292e-10,1.0,1.0,1.0


,tri_id,reason,suspect_count,cpu_hit,gpu_hit,cpu_normal_trace,gpu_hit_normal_trace,gpu_qef_normal_trace
0,9422592,largest_both_hit_normal_gap,0,True,True,1.0,1.0,1.0


,field,cpu_scalar,cpu_eigen,gpu_default,gpu_no_fma,gpu_fma,default_minus_cpu_eigen,no_fma_minus_cpu_eigen,fma_minus_cpu_eigen
0,v0.x,7.762512e-01,7.762512e-01,7.762512e-01,7.762512e-01,7.762512e-01,0.000000e+00,0.000000e+00,0.000000e+00
1,v0.y,7.296907e-01,7.296907e-01,7.296907e-01,7.296907e-01,7.296907e-01,0.000000e+00,0.000000e+00,0.000000e+00
2,v0.z,5.256871e-01,5.256871e-01,5.256871e-01,5.256871e-01,5.256871e-01,0.000000e+00,0.000000e+00,0.000000e+00
3,v1.x,7.780595e-01,7.780595e-01,7.780595e-01,7.780595e-01,7.780595e-01,0.000000e+00,0.000000e+00,0.000000e+00
4,v1.y,7.318841e-01,7.318841e-01,7.318841e-01,7.318841e-01,7.318841e-01,0.000000e+00,0.000000e+00,0.000000e+00
5,v1.z,5.259887e-01,5.259887e-01,5.259887e-01,5.259887e-01,5.259887e-01,0.000000e+00,0.000000e+00,0.000000e+00
6,v2.x,7.759653e-01,7.759653e-01,7.759653e-01,7.759653e-01,7.759653e-01,0.000000e+00,0.000000e+00,0.000000e+00
7,v2.y,7.299305e-01,7.299305e-01,7.299305e-01,7.299305e-01,7.299305e-01,0.000000e+00,0.000000e+00,0.000000e+00
8,v2.z,5.280434e-01,5.280434e-01,5.280434e-01,5.280434e-01,5.280434e-01,0.000000e+00,0.000000e+00,0.000000e+00
9,e0.x,1.808345e-03,1.808345e-03,1.808345e-03,1.808345e-03,1.808345e-03,0.000000e+00,0.000000e+00,0.000000e+00


## GPU Timing And Memory

Cold runs clear the caching allocator before the call. Warm results report the median across repeats. Memory numbers are peak deltas relative to the live inputs from `intersect_qef`.

In [9]:
def timed_cuda_call(fn, cold=False):
    if cold:
        torch.cuda.empty_cache()
    torch.cuda.synchronize()
    torch.cuda.reset_peak_memory_stats()
    base_alloc = torch.cuda.memory_allocated()
    base_reserved = torch.cuda.memory_reserved()
    start = torch.cuda.Event(enable_timing=True)
    end = torch.cuda.Event(enable_timing=True)
    start.record()
    out = fn()
    end.record()
    torch.cuda.synchronize()
    ms = start.elapsed_time(end)
    peak_alloc = torch.cuda.max_memory_allocated() - base_alloc
    peak_reserved = torch.cuda.max_memory_reserved() - base_reserved
    del out
    torch.cuda.synchronize()
    return {
        'ms': float(ms),
        'peak_alloc_mb': float(peak_alloc / 1024**2),
        'peak_reserved_mb': float(peak_reserved / 1024**2),
    }


def summarize_records(records):
    return {
        'ms_median': statistics.median(r['ms'] for r in records),
        'ms_min': min(r['ms'] for r in records),
        'peak_alloc_mb_median': statistics.median(r['peak_alloc_mb'] for r in records),
        'peak_reserved_mb_median': statistics.median(r['peak_reserved_mb'] for r in records),
    }


bench_rows = []
for input_name, case in INPUT_CASES.items():
    inter = INTERSECT_OUTPUTS[input_name]
    methods = {
        'old': lambda case=case, inter=inter: face_old(case, inter),
        'ref': lambda case=case, inter=inter: face_ref(case, inter),
        'new': lambda case=case, inter=inter: face_new(case, inter),
    }
    for method, fn in methods.items():
        cold = timed_cuda_call(fn, cold=True)
        _ = timed_cuda_call(fn, cold=False)
        warm_records = [timed_cuda_call(fn, cold=False) for _ in range(WARM_REPEATS)]
        warm = summarize_records(warm_records)
        bench_rows.append({
            'input': input_name,
            'triangles': case['triangles'],
            'voxels': int(inter.voxels.shape[0]),
            'input_triangle_mb': case['triangle_mb'],
            'method': method,
            'cold_ms': cold['ms'],
            'cold_peak_alloc_mb': cold['peak_alloc_mb'],
            'cold_peak_reserved_mb': cold['peak_reserved_mb'],
            **warm,
        })

show_rows(bench_rows)


,input,triangles,voxels,input_triangle_mb,method,cold_ms,cold_peak_alloc_mb,cold_peak_reserved_mb,ms_median,ms_min,peak_alloc_mb_median,peak_reserved_mb_median
0,original,280333,4676307,9.624470,old,31.090849,1552.552246,1108.0,25.762640,24.959999,1553.522461,0.0
1,original,280333,4676307,9.624470,ref,28.964064,178.387207,180.0,28.185440,27.556192,178.387207,0.0
2,original,280333,4676307,9.624470,new,4.573184,204.424805,182.0,2.789376,2.743744,204.424805,0.0
3,subdivided,10304077,4676306,353.762409,old,323.780609,17077.544922,19270.0,323.351486,321.073151,17077.544922,0.0
4,subdivided,10304077,4676306,353.762409,ref,17.439745,178.387207,180.0,15.984192,15.233024,178.387207,0.0
5,subdivided,10304077,4676306,353.762409,new,21.840897,561.791992,408.0,17.950400,16.954369,561.791992,0.0
